In [1]:
import pandas as pd
import re

In [2]:
f23 = 'p23v31_KMP_csv.csv'
f24 = 'p24v31_KMP_csv.csv'

df23 = pd.read_csv(f23, low_memory=False, encoding='cp949')
df24 = pd.read_csv(f24, low_memory=False, encoding='cp949')

for df in [df23, df24]:
    df.columns = [c.strip() for c in df.columns]

In [3]:
def pick(df, key):
    pats = [rf'^{key}$', rf'_{key}$', rf'{key}$']
    for p in pats:
        hits = [c for c in df.columns if re.search(p, c.lower())]
        if hits:
            return hits[0]
    raise KeyError(key)

In [4]:
keys = ['pid', 'age', 'gender', 'income', 'school', 'area', 'hhldsiz', 'job1', 'mar', 'a02014', 'a03008']

sel23 = {k: pick(df23, k) for k in keys}
sel24 = {k: pick(df24, k) for k in keys}

sub23 = df23[[sel23[k] for k in keys]].copy()
sub24 = df24[[sel24[k] for k in keys]].copy()

sub23.columns = ['pid'] + [f'p23_{k}' for k in keys[1:]]
sub24.columns = ['pid'] + [f'p24_{k}' for k in keys[1:]]

merged = sub23.merge(sub24[['pid', 'p24_a02014', 'p24_a03008']], on='pid', how='inner')

merged['churn'] = (
    ((merged['p23_a02014'] != merged['p24_a02014']) & merged['p23_a02014'].notna() & merged['p24_a02014'].notna()) |
    ((merged['p23_a03008'] != merged['p24_a03008']) & merged['p23_a03008'].notna() & merged['p24_a03008'].notna())
).astype('int8')

merged = merged[['pid'] + [f'p23_{k}' for k in keys[1:]] + ['churn']]
merged.to_csv('churn_panel_23_only.csv', index=False, encoding='utf-8-sig')

print(merged.shape)
print(merged.head())

(8277, 12)
     pid  p23_age  p23_gender  p23_income  p23_school  p23_area  p23_hhldsiz  \
0  10002        6           2           1           5         1            3   
1  10003        3           2           5           5         1            3   
2  10004        3           1           1           5         1            3   
3  20001        6           1           7           5         1            3   
4  20003        4           1           6           6         1            3   

   p23_job1  p23_mar p23_a02014 p23_a03008  churn  
0         2        2                     2      0  
1         1        1                     2      0  
2         2        1                     2      0  
3         1        2                     1      0  
4         1        1                     1      0  


In [5]:
churn_ratio = merged['churn'].mean()
print(churn_ratio)

0.3926543433611212


In [6]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix

# 데이터 로드
df = pd.read_csv('churn_panel_23_only.csv', encoding='utf-8-sig')

# 타깃/특성 분리
X = df.drop(columns=['churn'])
y = df['churn']

# 수치형/범주형 컬럼 지정
num_cols = ['p23_age', 'p23_income', 'p23_hhldsiz']
cat_cols = [c for c in X.columns if c not in num_cols + ['pid']]

# 학습/평가 분리
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 전처리
num_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

cat_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocess = ColumnTransformer([
    ('num', num_pipe, num_cols),
    ('cat', cat_pipe, cat_cols)
], remainder='drop')

# 모델
model = LogisticRegression(max_iter=2000, class_weight='balanced')

clf = Pipeline([
    ('preprocess', preprocess),
    ('model', model)
])

# 학습
clf.fit(X_train, y_train)

# 예측 및 평가
pred = clf.predict(X_test)
proba = clf.predict_proba(X_test)[:, 1]

print(classification_report(y_test, pred))
print('ROC-AUC:', roc_auc_score(y_test, proba))
print('Confusion matrix:\n', confusion_matrix(y_test, pred))

              precision    recall  f1-score   support

           0       0.70      0.57      0.63      1006
           1       0.48      0.62      0.54       650

    accuracy                           0.59      1656
   macro avg       0.59      0.60      0.59      1656
weighted avg       0.62      0.59      0.60      1656

ROC-AUC: 0.652034714788194
Confusion matrix:
 [[576 430]
 [246 404]]


In [7]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression

plt.style.use('seaborn-v0_8-whitegrid')

df = pd.read_csv('churn_panel_23_only.csv', encoding='utf-8-sig')

X = df.drop(columns=['churn'])
y = df['churn']

num_cols = ['p23_age', 'p23_income', 'p23_hhldsiz']
cat_cols = [c for c in X.columns if c not in num_cols + ['pid']]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

preprocess = ColumnTransformer([
    ('num', Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ]), num_cols),
    ('cat', Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore'))
    ]), cat_cols)
], remainder='drop')

clf = Pipeline([
    ('preprocess', preprocess),
    ('model', LogisticRegression(max_iter=2000, class_weight='balanced'))
])

clf.fit(X_train, y_train)

feature_names = clf.named_steps['preprocess'].get_feature_names_out()
coefs = clf.named_steps['model'].coef_.ravel()

imp = pd.DataFrame({
    'feature': feature_names,
    'importance': np.abs(coefs),
    'signed': coefs
})

imp = imp.sort_values('importance', ascending=False).head(20).reset_index(drop=True)
imp['feature'] = (
    imp['feature']
    .str.replace('num__', '', regex=False)
    .str.replace('cat__', '', regex=False)
    .str.replace('p23_', '', regex=False)
)

imp.to_csv('model_feature_importance_top20_matplotlib.csv', index=False, encoding='utf-8-sig')

plot_df = imp.sort_values('importance', ascending=True)

fig, ax = plt.subplots(figsize=(10, 8))
ax.barh(plot_df['feature'], plot_df['importance'], color='#4C78A8')
ax.set_title('모델 변수 중요도 상위 20개 (2023)')
ax.set_xlabel('Importance')
ax.set_ylabel('Feature')

for i, v in enumerate(plot_df['importance']):
    ax.text(v, i, f'{v:.3f}', va='center', ha='left', fontsize=9)

plt.tight_layout()
plt.savefig('feature_importance_top20_matplotlib.png', dpi=200, bbox_inches='tight')
plt.close()

C:\Users\HJP\AppData\Local\Temp\ipykernel_30908\3715532454.py:73: UserWarning: Glyph 47784 (\N{HANGUL SYLLABLE MO}) missing from font(s) Arial.
  plt.tight_layout()
C:\Users\HJP\AppData\Local\Temp\ipykernel_30908\3715532454.py:73: UserWarning: Glyph 45944 (\N{HANGUL SYLLABLE DEL}) missing from font(s) Arial.
  plt.tight_layout()
C:\Users\HJP\AppData\Local\Temp\ipykernel_30908\3715532454.py:73: UserWarning: Glyph 48320 (\N{HANGUL SYLLABLE BYEON}) missing from font(s) Arial.
  plt.tight_layout()
C:\Users\HJP\AppData\Local\Temp\ipykernel_30908\3715532454.py:73: UserWarning: Glyph 49688 (\N{HANGUL SYLLABLE SU}) missing from font(s) Arial.
  plt.tight_layout()
C:\Users\HJP\AppData\Local\Temp\ipykernel_30908\3715532454.py:73: UserWarning: Glyph 51473 (\N{HANGUL SYLLABLE JUNG}) missing from font(s) Arial.
  plt.tight_layout()
C:\Users\HJP\AppData\Local\Temp\ipykernel_30908\3715532454.py:73: UserWarning: Glyph 50836 (\N{HANGUL SYLLABLE YO}) missing from font(s) Arial.
  plt.tight_layout()
C:\U

In [8]:
imp

,feature,importance,signed
0,area_16,1.018693,-1.018693
1,area_13,0.967125,0.967125
2,area_9,0.949526,-0.949526
3,a02014_4,0.874398,0.874398
4,area_12,0.756553,0.756553
5,area_3,0.755011,-0.755011
6,a02014_,0.702490,-0.702490
7,area_5,0.638623,0.638623
8,area_17,0.543057,0.543057
9,a02014_3,0.466202,0.466202
